# 02 · Transform
**Purpose:** Clean, validate, and enrich the raw API data before loading.  
All transform functions are **pure** — no DB calls, no side effects, fully testable.

---
### Transformations applied
| Step | What happens |
|---|---|
| Null handling | Missing prices / percentages default to `0.0` |
| Type casting | Prices → `float`, market caps → `int` |
| Rounding | Prices to 6 dp, percentages to 4 dp |
| Timestamp normalization | All datetimes → UTC ISO-8601 |
| Derived: `volatility_score` | `abs(price_change_1h − price_change_24h)` |
| Schema flattening | Nested API fields → flat dict ready for DB |


In [ ]:
import logging
from datetime import datetime, timezone
from typing import Optional

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)
print("✓ Imports ready")


---
## Step 1 — Re-run the extract
*(Skip this cell if `raw_market_data` is already in memory from notebook 01)*


In [ ]:
import requests

BASE_URL = "https://api.coingecko.com/api/v3"
COINS = [
    "bitcoin", "ethereum", "solana", "cardano",
    "polkadot", "chainlink", "avalanche-2", "uniswap"
]

def fetch_market_data(coins=COINS):
    params = {
        "vs_currency": "usd", "ids": ",".join(coins),
        "order": "market_cap_desc", "per_page": len(coins), "page": 1,
        "sparkline": "false", "price_change_percentage": "1h,24h,7d"
    }
    r = requests.get(f"{BASE_URL}/coins/markets", params=params, timeout=15)
    r.raise_for_status()
    return r.json()

def fetch_global_stats():
    r = requests.get(f"{BASE_URL}/global", timeout=15)
    r.raise_for_status()
    return r.json().get("data", {})

raw_market_data = fetch_market_data()
raw_global = fetch_global_stats()
print(f"✓ Fetched {len(raw_market_data)} coin records + global stats")


---
## Step 2 — `transform_market_data()`


In [ ]:
def transform_market_data(raw_records: list) -> list:
    """
    Clean and reshape raw CoinGecko market records into a flat,
    database-ready schema.
    """
    if not raw_records:
        logger.warning("No records to transform")
        return []

    transformed = []
    ingested_at = datetime.now(timezone.utc).isoformat()

    for record in raw_records:
        try:
            price_change_1h  = record.get("price_change_percentage_1h_in_currency")  or 0.0
            price_change_24h = record.get("price_change_percentage_24h_in_currency") or 0.0
            price_change_7d  = record.get("price_change_percentage_7d_in_currency")  or 0.0

            # Derived metric: absolute swing between 1h and 24h = short-term volatility proxy
            volatility_score = round(abs(price_change_1h - price_change_24h), 4)

            last_updated_raw = record.get("last_updated")
            last_updated = (
                datetime.fromisoformat(last_updated_raw.replace("Z", "+00:00")).isoformat()
                if last_updated_raw else ingested_at
            )

            cleaned = {
                "coin_id":                     record.get("id", "unknown"),
                "symbol":                      (record.get("symbol") or "").upper(),
                "name":                        record.get("name", "Unknown"),
                "current_price_usd":           round(float(record.get("current_price") or 0), 6),
                "market_cap_usd":              int(record.get("market_cap") or 0),
                "market_cap_rank":             record.get("market_cap_rank"),
                "fully_diluted_valuation_usd": record.get("fully_diluted_valuation"),
                "total_volume_24h_usd":        int(record.get("total_volume") or 0),
                "high_24h_usd":                round(float(record.get("high_24h") or 0), 6),
                "low_24h_usd":                 round(float(record.get("low_24h") or 0), 6),
                "price_change_24h_usd":        round(float(record.get("price_change_24h") or 0), 6),
                "price_change_pct_1h":         round(price_change_1h, 4),
                "price_change_pct_24h":        round(price_change_24h, 4),
                "price_change_pct_7d":         round(price_change_7d, 4),
                "volatility_score":            volatility_score,
                "circulating_supply":          record.get("circulating_supply"),
                "total_supply":                record.get("total_supply"),
                "max_supply":                  record.get("max_supply"),
                "ath_usd":                     round(float(record.get("ath") or 0), 6),
                "ath_change_pct":              round(float(record.get("ath_change_percentage") or 0), 4),
                "atl_usd":                     round(float(record.get("atl") or 0), 10),
                "last_updated":                last_updated,
                "ingested_at":                 ingested_at,
            }
            transformed.append(cleaned)

        except (TypeError, ValueError, KeyError) as e:
            logger.warning(f"Skipping malformed record for {record.get('id', 'UNKNOWN')}: {e}")
            continue

    logger.info(f"Transformed {len(transformed)}/{len(raw_records)} records")
    return transformed


In [ ]:
clean_market_data = transform_market_data(raw_market_data)

print(f"Input records  : {len(raw_market_data)}")
print(f"Output records : {len(clean_market_data)}")
print(f"Output schema  : {len(clean_market_data[0])} fields")
print(f"\nSchema fields:")
for k, v in clean_market_data[0].items():
    print(f"  {k:<35} {repr(v)}")


---
## Step 3 — Inspect derived fields


In [ ]:
print(f"{'Coin':<14} {'1h %':>8} {'24h %':>8} {'Volatility':>12}  Interpretation")
print("-" * 65)
for r in clean_market_data:
    interp = "stable"
    if r["volatility_score"] > 2.0:
        interp = "⚠ high divergence"
    elif r["volatility_score"] > 0.8:
        interp = "↑ moderate"
    print(
        f"  {r['name']:<14} "
        f"{r['price_change_pct_1h']:>7.3f}% "
        f"{r['price_change_pct_24h']:>7.3f}% "
        f"{r['volatility_score']:>12.4f}  {interp}"
    )


In [ ]:
# ── Verify timestamp normalization ──────────────────────────────
print("Timestamp normalization check:")
print(f"  ingested_at  : {clean_market_data[0]['ingested_at']}")
print(f"  last_updated : {clean_market_data[0]['last_updated']}")
print()

# ── Verify null safety ────────────────────────────────────────────
import copy
bad = copy.deepcopy(raw_market_data[0])
bad["current_price"] = None
bad["price_change_percentage_1h_in_currency"] = None

result = transform_market_data([bad])
print("Null safety check (None inputs):")
print(f"  current_price_usd  → {result[0]['current_price_usd']}  (expected 0.0)")
print(f"  price_change_pct_1h → {result[0]['price_change_pct_1h']}  (expected 0.0)")
print(f"  volatility_score   → {result[0]['volatility_score']}")


---
## Step 4 — `transform_global_stats()`


In [ ]:
def transform_global_stats(raw_stats: dict) -> Optional[dict]:
    """Clean and flatten global market statistics."""
    if not raw_stats:
        return None
    try:
        total_market_cap = raw_stats.get("total_market_cap", {})
        total_volume     = raw_stats.get("total_volume", {})
        market_cap_pct   = raw_stats.get("market_cap_percentage", {})

        return {
            "total_market_cap_usd":      int(total_market_cap.get("usd") or 0),
            "total_volume_24h_usd":      int(total_volume.get("usd") or 0),
            "btc_dominance_pct":         round(float(market_cap_pct.get("btc") or 0), 4),
            "eth_dominance_pct":         round(float(market_cap_pct.get("eth") or 0), 4),
            "active_cryptocurrencies":   raw_stats.get("active_cryptocurrencies"),
            "total_exchanges":           raw_stats.get("markets"),
            "market_cap_change_pct_24h": round(float(raw_stats.get("market_cap_change_percentage_24h_usd") or 0), 4),
            "ingested_at":               datetime.now(timezone.utc).isoformat(),
        }
    except (TypeError, ValueError) as e:
        logger.error(f"Failed to transform global stats: {e}")
        return None


In [ ]:
clean_global = transform_global_stats(raw_global)

print("Cleaned global stats:")
print("-" * 45)
for k, v in clean_global.items():
    if isinstance(v, int) and v > 1_000_000:
        print(f"  {k:<35} ${v:,.0f}")
    elif isinstance(v, float):
        print(f"  {k:<35} {v:.4f}%")
    else:
        print(f"  {k:<35} {v}")


---
## ✅ Transform complete

`clean_market_data` and `clean_global` are ready.  
Pass them into **03_load.ipynb** to write to the SQLite database.
